In [1]:
import pandas as pd
import numpy as np
import warnings

# Suppress pandas future warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='pandas')
pd.set_option('future.no_silent_downcasting', True)

In [2]:
df = pd.read_csv("train.csv")

In [3]:
df

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


# Exploratory Data Analysis (EDA)

In [4]:
# Check basic info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [5]:
# Summary statistics
df.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [6]:
# Check for missing values
df.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [7]:
# Value counts for categorical columns
categorical_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Transported']
for col in categorical_cols:
    print(f"Value counts for {col}:")
    print(df[col].value_counts())
    print("\n")

Value counts for HomePlanet:
HomePlanet
Earth     4602
Europa    2131
Mars      1759
Name: count, dtype: int64


Value counts for CryoSleep:
CryoSleep
False    5439
True     3037
Name: count, dtype: int64


Value counts for Destination:
Destination
TRAPPIST-1e      5915
55 Cancri e      1800
PSO J318.5-22     796
Name: count, dtype: int64


Value counts for VIP:
VIP
False    8291
True      199
Name: count, dtype: int64


Value counts for Transported:
Transported
True     4378
False    4315
Name: count, dtype: int64




# Data Preprocessing

In [5]:
# Make a copy of the dataframe
df_processed = df.copy()

In [6]:
# Fill missing values
# Numerical columns
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_cols:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].fillna(df_processed[col].median())

# Categorical columns
cat_cols = ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']
for col in cat_cols:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].fillna(df_processed[col].mode()[0])

In [7]:
# Split Cabin into deck, num, side
df_processed[['Deck', 'Num', 'Side']] = df_processed['Cabin'].str.split('/', expand=True)
df_processed['Num'] = pd.to_numeric(df_processed['Num'], errors='coerce')
df_processed['Num'] = df_processed['Num'].fillna(df_processed['Num'].median())
df_processed.drop('Cabin', axis=1, inplace=True)

In [8]:
# Split PassengerId into group and person
df_processed[['Group', 'Person']] = df_processed['PassengerId'].str.split('_', expand=True)
df_processed['Group'] = pd.to_numeric(df_processed['Group'])
df_processed['Person'] = pd.to_numeric(df_processed['Person'])
df_processed.drop('PassengerId', axis=1, inplace=True)

In [9]:
# Drop Name column
df_processed.drop('Name', axis=1, inplace=True)

In [10]:
# Encode categorical variables
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
cat_cols_to_encode = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
for col in cat_cols_to_encode:
    df_processed[col] = le.fit_transform(df_processed[col])

In [11]:
# Check for any remaining missing values
print("Missing values in processed train data:")
print(df_processed.isnull().sum())

Missing values in processed train data:
HomePlanet      0
CryoSleep       0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Transported     0
Deck            0
Num             0
Side            0
Group           0
Person          0
dtype: int64


# Model Training

In [12]:
# Import necessary libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Prepare features and target
X = df_processed.drop('Transported', axis=1)
y = df_processed['Transported']

# Split the data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Train the model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [14]:
# Make predictions on validation set
y_pred = model.predict(X_val)

# Evaluate the model
accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy: {accuracy}")
print(classification_report(y_val, y_pred))

Accuracy: 0.7958596894767107
              precision    recall  f1-score   support

       False       0.78      0.81      0.80       861
        True       0.81      0.78      0.79       878

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739



# Predict on Test Data

In [15]:
# Load test data
df_test = pd.read_csv("test.csv")
passenger_ids = df_test['PassengerId']  # Save for submission

In [16]:
# Preprocess test data (same as train)
df_test_processed = df_test.copy()

# Fill missing values using train medians/modes
for col in num_cols:
    df_test_processed[col] = df_test_processed[col].fillna(df[col].median())
for col in ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']:
    df_test_processed[col] = df_test_processed[col].fillna(df[col].mode()[0])

# Split Cabin
df_test_processed[['Deck', 'Num', 'Side']] = df_test_processed['Cabin'].str.split('/', expand=True)
df_test_processed['Num'] = pd.to_numeric(df_test_processed['Num'], errors='coerce')
df_test_processed['Num'] = df_test_processed['Num'].fillna(df_processed['Num'].median())
df_test_processed.drop('Cabin', axis=1, inplace=True)

# Split PassengerId
df_test_processed[['Group', 'Person']] = df_test_processed['PassengerId'].str.split('_', expand=True)
df_test_processed['Group'] = pd.to_numeric(df_test_processed['Group'])
df_test_processed['Person'] = pd.to_numeric(df_test_processed['Person'])
df_test_processed.drop('PassengerId', axis=1, inplace=True)

# Drop Name
df_test_processed.drop('Name', axis=1, inplace=True)

# Encode using the same label encoders
for col in cat_cols_to_encode:
    if col in df_test_processed.columns:
        df_test_processed[col] = pd.Categorical(df_test_processed[col], categories=df_processed[col].unique()).codes
        # Handle unseen categories by replacing -1 with the mode code
        if (df_test_processed[col] == -1).any():
            mode_code = df_processed[col].mode()[0]
            df_test_processed[col] = df_test_processed[col].replace(-1, mode_code)

In [17]:
# Check for any remaining missing values in test
print("Missing values in processed test data:")
print(df_test_processed.isnull().sum())

Missing values in processed test data:
HomePlanet      0
CryoSleep       0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Deck            0
Num             0
Side            0
Group           0
Person          0
dtype: int64


In [18]:
# Predict on test data
test_predictions = model.predict(df_test_processed)

# Create submission dataframe
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Transported': test_predictions
})

# Save to CSV
submission.to_csv("sample_submission.csv", index=False)
print("Submission saved to sample_submission.csv")

Submission saved to sample_submission.csv
